In [7]:
#!python VideoSelectionHelper.py

Set Paths for Video and Configuration Files


VIDEO_PATH = r"C:\Users\evely\OneDrive\Documents\2025-2026\PneumaticActuators\RibbedActuators\PrintQualityTest\Analysis\0.12-S4\Print2\Round2\0.12-S4-P2-R2-Cleaned.mp4"
CONFIG_PATH = r"C:\Users\evely\OneDrive\Documents\2025-2026\PneumaticActuators\RibbedActuators\PrintQualityTest\Analysis\0.12-S4\Print2\Round2\0.12-S4-P2-R2_config.json"

from datetime import datetime
CSV_FOLDER = r"C:\Users\evely\OneDrive\Documents\2025-2026\PneumaticActuators\RibbedActuators\PrintQualityTest\Analysis\0.12-S4\Print2\Round3"
CSV_PATH = f"{CSV_FOLDER}\\analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"


In [8]:
from tkinter import Tk, filedialog
from datetime import datetime
import os
import sys

def get_paths(default_video=None, default_config=None, default_csv_folder=None):
    """
    Prompts the user to select:
      - a video file (required)
      - a config JSON file (required; dialog repeats if cancelled)
      - a CSV output folder (required)
    Returns: (VIDEO_PATH, CONFIG_PATH, CSV_PATH)
    Exits the program if required selections are not made.
    """
    root = Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    # Video selection (required)
    video_path = filedialog.askopenfilename(
        title="Select video file",
        initialdir=os.path.dirname(default_video) if default_video else None,
        filetypes=[("Video files", "*.mp4 *.avi *.mov *.mkv"), ("All files", "*.*")]
    )
    if not video_path:
        root.destroy()
        sys.exit("No video selected. Exiting.")

    # Config selection (required) - keep asking until a file is chosen or user closes the dialog
    config_path = None
    while not config_path:
        config_path = filedialog.askopenfilename(
            title="Select config JSON (required)",
            initialdir=os.path.dirname(default_config) if default_config else None,
            filetypes=[("JSON files", "*.json"), ("All files", "*.*")]
        )
        if not config_path:
            # User cancelled; ask whether they want to try again or exit
            try_again = filedialog.messagebox.askyesno(
                "Config required",
                "A configuration file is required. Would you like to choose a file?"
            )
        else:
            break
        if not try_again:
            root.destroy()
            sys.exit("Config not selected. Exiting.")
        config_path = None  # loop again

    # CSV folder selection (required)
    csv_folder = filedialog.askdirectory(
        title="Select folder to save CSV",
        initialdir=default_csv_folder or os.path.dirname(video_path)
    )
    if not csv_folder:
        root.destroy()
        sys.exit("No CSV folder selected. Exiting.")

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_filename = f"analysis_{timestamp}.csv"
    csv_path = os.path.join(csv_folder, csv_filename)

    root.destroy()
    return video_path, config_path, csv_path

# Example usage:
# VIDEO_PATH, CONFIG_PATH, CSV_PATH = get_paths()

In [9]:
VIDEO_PATH, CONFIG_PATH, CSV_PATH = get_paths()

In [10]:
MAX_PRESSURE = 700  # KpA
exclusion_radius = 60 # pixels

In [11]:
def resize_window_preserve_aspect(window_name, frame, max_width=2560, max_height=1080):
    h, w = frame.shape[:2]
    scale = min(max_width / w, max_height / h, 1.0)
    new_w, new_h = int(w * scale), int(h * scale)
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, new_w, new_h)

# Analysis

In [12]:
import cv2
import json
import math
import numpy as np
import csv
import os
from collections import deque

# -------------------------
# Helper: load config
# -------------------------
def load_config(path):
    with open(path, "r") as f:
        return json.load(f)

# -------------------------
# Geometry / angle helpers
# -------------------------
def signed_acute_angle(base, tip):
    dx = tip["x"] - base["x"]
    dy = tip["y"] - base["y"]
    theta_deg = math.degrees(math.atan2(dy, dx))
    theta_abs = abs(theta_deg) % 180
    acute = theta_abs if theta_abs <= 90 else 180 - theta_abs
    sign = 1 if tip["y"] > base["y"] else -1
    return acute * sign

def compute_angle(p1, p2):
    dx, dy = p2[0]-p1[0], p2[1]-p1[1]
    return math.degrees(math.atan2(dy, dx))

def normalize_angle(angle):
    return (angle + 360) % 360

# -------------------------
# Actuator tip detection
# -------------------------
def find_actuator_tip(roi_frame, ax, ay, base, lower_red1, upper_red1, lower_red2, upper_red2):
    """
    roi_frame: BGR image cropped to actuator ROI
    ax, ay: top-left coordinates of ROI in full frame (used to convert local->global)
    base: tuple (bx, by) in full-frame coordinates
    returns: (x,y) global coordinates of detected tip or None
    """
    if roi_frame is None or roi_frame.size == 0:
        return None

    hsv = cv2.cvtColor(roi_frame, cv2.COLOR_BGR2HSV)
    mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
    mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
    mask = cv2.bitwise_or(mask1, mask2)

    kernel = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    valid = [c for c in contours if cv2.contourArea(c) > 50]  # filter small blobs

    best_tip, best_dist = None, 0
    for c in valid:
        M = cv2.moments(c)
        if M.get("m00", 0) == 0:
            continue
        cx_local = int(M["m10"]/M["m00"])
        cy_local = int(M["m01"]/M["m00"])
        candidate_tip = (cx_local + ax, cy_local + ay)
        dist = math.hypot(candidate_tip[0]-base[0], candidate_tip[1]-base[1])
        if dist > best_dist:
            best_dist, best_tip = dist, candidate_tip
    return best_tip

# -------------------------
# Window resizing helper
# -------------------------
def resize_window_preserve_aspect(window_name, frame, max_width=1000, max_height=700):
    h, w = frame.shape[:2]
    scale = min(max_width / w, max_height / h, 1.0)
    new_w, new_h = int(w * scale), int(h * scale)
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, new_w, new_h)

# -------------------------
# Main analysis
# -------------------------
def main():
    # Required globals (set these before running)
    required = ("CONFIG_PATH", "VIDEO_PATH", "CSV_PATH", "exclusion_radius")
    for name in required:
        if name not in globals():
            raise RuntimeError(f"Required global not set: {name}")

    cfg = load_config(CONFIG_PATH)

    # Actuator base point (required)
    if "angle_base_point" not in cfg or not isinstance(cfg["angle_base_point"], dict):
        raise KeyError("Config missing 'angle_base_point'")
    base = (int(cfg["angle_base_point"]["x"]), int(cfg["angle_base_point"]["y"]))

    # Actuator ROI: accept either top_left/bottom_right or x,y,w,h
    if "actuator_roi" not in cfg or not isinstance(cfg["actuator_roi"], dict):
        raise KeyError("Config missing 'actuator_roi'")

    aro = cfg["actuator_roi"]
    # prefer top_left/bottom_right if present
    if "top_left" in aro and "bottom_right" in aro:
        atl = aro["top_left"]
        abr = aro["bottom_right"]
        ax = int(atl["x"]); ay = int(atl["y"])
        aw = int(abr["x"] - ax); ah = int(abr["y"] - ay)
    elif all(k in aro for k in ("x","y","w","h")):
        ax = int(aro["x"]); ay = int(aro["y"])
        aw = int(aro["w"]); ah = int(aro["h"])
    else:
        raise KeyError("actuator_roi must contain either top_left/bottom_right or x,y,w,h")

    if aw <= 0 or ah <= 0:
        raise ValueError(f"Parsed actuator ROI has non-positive size: ({aw},{ah})")

    # Print summary
    print(f"Base: {base}")
    print(f"Actuator ROI: x={ax}, y={ay}, w={aw}, h={ah}")

    # Color thresholds for red (tune if needed)
    lower_red1, upper_red1 = np.array([0,120,120]), np.array([10,255,255])
    lower_red2, upper_red2 = np.array([170,120,120]), np.array([180,255,255])

    # Buffers and smoothing
    tip_buffer = deque(maxlen=5)
    last_angle = None

    # Video capture
    cap = cv2.VideoCapture(VIDEO_PATH)
    ret, frame = cap.read()
    if not ret or frame is None:
        raise RuntimeError("Could not read first frame from video")
    resize_window_preserve_aspect("Analysis", frame)

    # CSV setup
    os.makedirs(os.path.dirname(CSV_PATH) or ".", exist_ok=True)
    try:
        csvfile = open(CSV_PATH, "w", newline="")
    except Exception as e:
        raise RuntimeError(f"Unable to open CSV path '{CSV_PATH}': {e}")
    writer = csv.writer(csvfile)
    writer.writerow(["Frame", "ActuatorAngle_deg"])

    frame_idx = 0
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_idx += 1

            # Crop actuator ROI safely
            y0 = max(0, ay); x0 = max(0, ax)
            y1 = min(frame.shape[0], y0 + max(0, ah))
            x1 = min(frame.shape[1], x0 + max(0, aw))
            roi_frame = frame[y0:y1, x0:x1]

            tip_global = None
            if roi_frame.size > 0:
                tip = find_actuator_tip(roi_frame, x0, y0, base,
                                        lower_red1, upper_red1, lower_red2, upper_red2)
                if tip:
                    # enforce exclusion radius from base
                    if math.hypot(tip[0]-base[0], tip[1]-base[1]) >= exclusion_radius:
                        tip_buffer.append(tip)
            if tip_buffer:
                tip_global = tuple(np.mean(tip_buffer, axis=0).astype(int))

            # Compute actuator angle if tip found
            actuator_angle = None
            if tip_global:
                base_dict = {"x": base[0], "y": base[1]}
                tip_dict = {"x": tip_global[0], "y": tip_global[1]}
                actuator_angle = signed_acute_angle(base_dict, tip_dict)
                last_angle = actuator_angle

                # Draw overlays
                cv2.circle(frame, base, 6, (0,0,255), -1)
                cv2.circle(frame, tip_global, 6, (0,255,0), -1)
                cv2.line(frame, base, tip_global, (255,255,0), 2)
                cv2.putText(frame, f"Actuator Angle: {actuator_angle:.1f} deg", (20,40),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)
            else:
                # still draw base
                cv2.circle(frame, base, 6, (0,0,255), -1)
                if last_angle is not None:
                    cv2.putText(frame, f"Actuator Angle (last): {last_angle:.1f} deg", (20,40),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)

            # CSV write (only actuator angle). If missing, write NaN
            try:
                pval = actuator_angle if actuator_angle is not None else float("nan")
                writer.writerow([frame_idx, pval])
                if frame_idx % 100 == 0:
                    csvfile.flush()
            except Exception as e:
                print("CSV write error:", e)

            cv2.imshow("Analysis", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    finally:
        cap.release()
        csvfile.close()
        cv2.destroyAllWindows()
        print(f"CSV export complete: {os.path.abspath(CSV_PATH)}")

# -------------------------
# Run guard
# -------------------------
if __name__ == "__main__":
    main()

Base: (1254, 702)
Actuator ROI: x=496, y=430, w=1016, h=1150
CSV export complete: C:\Users\ewatt015\OneDrive\Documents\2025-2026\PneumaticActuators\Code\analysis_20251211_181835.csv


In [13]:
cfg = load_config(CONFIG_PATH)
import pprint
pprint.pprint(cfg)

{'actuator_roi': {'h': 1150, 'w': 1016, 'x': 496, 'y': 430},
 'angle_base_point': {'x': 1254, 'y': 702},
 'angle_tip_point': {'x': 696, 'y': 876}}
